# Radar Tutor UC · Prueba de concepto

**Ciencia de Datos · Unidad 2 (Learning Analytics) · Trabajo aplicativo · Universidad de Cuenca**

Este notebook demuestra que el **indicador central** de Radar Tutor UC, la *probabilidad de riesgo académico al cierre de la semana 4* (`I5`), se puede calcular con datos de interacción como los de eVirtual (Moodle). Con esos datos se construyen las cuatro pantallas diseñadas en los wireframes.

| Elemento del diseño | Dónde se usa en este notebook |
|---|---|
| Pregunta analítica | *¿Qué estudiantes de un curso tienen alta probabilidad de reprobar o abandonar la asignatura, según su actividad en eVirtual y sus entregas durante las cuatro primeras semanas?* |
| Diccionario de datos (`02_datos/Diccionario_de_datos.pdf`) | Sección 2: cálculo de `I1`–`I6` |
| Wireframes P1–P4 (`03_diseno/`) | Sección 4: las cuatro pantallas construidas con datos reales del dataset |
| Plan de validación (`06_validacion/Plan_de_validacion.pdf`) | Sección 6: lo que este prototipo todavía no puede responder |

**Contenido**
1. Carga y descripción del dataset
2. Limpieza y construcción de los indicadores
3. Modelo base de riesgo y su desempeño
4. Prototipo de la herramienta: pantallas P1 a P4
5. Revisión de sesgos
6. Lo que este prototipo no puede decir todavía

**Dataset.** Usamos OULAD (*Open University Learning Analytics Dataset*; Kuzilek, Hlosta y Zdrahal, 2017, *Scientific Data* 4:170171), con licencia CC BY 4.0. No tenemos acceso a los datos de eVirtual, y este es el dataset público más parecido a los registros de Moodle. La primera ejecución lo descarga del repositorio UCI (unos 45 MB) y tarda alrededor de 2 minutos.

In [1]:
# Configuración general
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve, recall_score, precision_score

pd.set_option("display.precision", 3)
pio.templates.default = "plotly_white"
pio.templates["plotly_white"].layout.separators = ",."

def dec(x, n=2):
    """Número con coma decimal, para textos y tablas."""
    return f"{x:.{n}f}".replace(".", ",")

SEMILLA = 42
CORTE_DIAS = 28                      # la alerta se emite al cerrar la semana 4
UMBRAL_ROJO, UMBRAL_AMARILLO = 0.60, 0.40

# Paleta común a todas las figuras (la misma de los wireframes)
COLOR = {"Rojo": "#d62728", "Amarillo": "#e0a800", "Verde": "#2ca02c",
         "azul": "#1f4e79", "gris": "#7f8c8d", "fondo": "#f4f6f9", "borde": "#d0d7e2"}
FUENTE = dict(family="Segoe UI, Helvetica, Arial, sans-serif", color="#1f2933")

URL_OULAD = "https://archive.ics.uci.edu/static/public/349/open+university+learning+analytics+dataset.zip"
DATA_DIR = Path("data/oulad")

if not (DATA_DIR / "studentVle.csv").exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print("Descargando OULAD (~45 MB)...")
    zip_path, _ = urllib.request.urlretrieve(URL_OULAD, DATA_DIR / "oulad.zip")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA_DIR)
print("Archivos disponibles:", ", ".join(sorted(p.name for p in DATA_DIR.glob("*.csv"))))

Archivos disponibles: assessments.csv, courses.csv, studentAssessment.csv, studentInfo.csv, studentRegistration.csv, studentVle.csv, vle.csv


## 1. Carga y descripción del dataset

Las tablas de OULAD se relacionan por `code_module` (asignatura), `code_presentation` (período: año más `B` = febrero o `J` = octubre) e `id_student` (seudónimo). La tabla siguiente indica a qué fuente de la UCuenca equivale cada una.

In [2]:
K = ["code_module", "code_presentation", "id_student"]

courses = pd.read_csv(DATA_DIR / "courses.csv")
info = pd.read_csv(DATA_DIR / "studentInfo.csv", na_values="?")
reg = pd.read_csv(DATA_DIR / "studentRegistration.csv", na_values="?")
assessments = pd.read_csv(DATA_DIR / "assessments.csv", na_values="?")
st_assess = pd.read_csv(DATA_DIR / "studentAssessment.csv", na_values="?")
vle_sites = pd.read_csv(DATA_DIR / "vle.csv", na_values="?")
# studentVle tiene ~10 millones de filas: se cargan solo las columnas necesarias con tipos compactos
st_vle = pd.read_csv(DATA_DIR / "studentVle.csv", usecols=K + ["date", "sum_click"],
                     dtype={"code_module": "category", "code_presentation": "category",
                            "id_student": "int32", "date": "int16", "sum_click": "int32"})

descripcion = pd.DataFrame([
    ("courses", courses, "Asignatura × período y su duración en días", "Oferta académica"),
    ("studentInfo", info, "Datos demográficos, intentos previos, créditos y resultado final", "Matrícula e historial"),
    ("studentRegistration", reg, "Fechas de matrícula y de retiro", "Matrícula y retiros"),
    ("assessments", assessments, "Evaluaciones con fecha límite y peso", "Tareas de eVirtual"),
    ("studentAssessment", st_assess, "Entregas: fecha de envío y nota", "Entregas y calificaciones"),
    ("vle", vle_sites, "Recursos del aula virtual", "Recursos de Moodle"),
    ("studentVle", st_vle, "Clics diarios por estudiante y recurso", "Registro de actividad (logs)"),
], columns=["tabla", "df", "contenido", "equivalente UCuenca"])
descripcion["filas"] = descripcion.df.map(len)
descripcion["columnas"] = descripcion.df.map(lambda d: d.shape[1])
display(descripcion.drop(columns="df").set_index("tabla").style.format({"filas": "{:,}"}))

resumen = pd.Series({
    "Estudiantes únicos": f"{info.id_student.nunique():,}",
    "Matrículas (estudiante × curso)": f"{len(info):,}",
    "Asignaturas": courses.code_module.nunique(),
    "Cursos (asignatura × período)": len(courses),
    "Períodos": ", ".join(sorted(courses.code_presentation.unique())),
    "Registros de clics diarios": f"{len(st_vle):,}",
}, name="valor")
display(resumen.to_frame())

,contenido,equivalente UCuenca,filas,columnas
tabla,,,,
courses,Asignatura × período y su duración en días,Oferta académica,22,3
studentInfo,"Datos demográficos, intentos previos, créditos y resultado final",Matrícula e historial,"32,593",12
studentRegistration,Fechas de matrícula y de retiro,Matrícula y retiros,"32,593",5
assessments,Evaluaciones con fecha límite y peso,Tareas de eVirtual,206,6
studentAssessment,Entregas: fecha de envío y nota,Entregas y calificaciones,"173,912",5
vle,Recursos del aula virtual,Recursos de Moodle,"6,364",6
studentVle,Clics diarios por estudiante y recurso,Registro de actividad (logs),"10,655,280",5


,valor
Estudiantes únicos,"28,785"
Matrículas (estudiante × curso),"32,593"
Asignaturas,7
Cursos (asignatura × período),22
Períodos,"2013B, 2013J, 2014B, 2014J"
Registros de clics diarios,"10,655,280"


## 2. Limpieza y construcción de los indicadores

### 2.1 Población y variable objetivo

- **Momento de la alerta:** fin de la semana 4 (día < 28). En ese punto el tutor todavía tiene margen para intervenir antes de los primeros parciales.
- **Población:** matrículas que siguen activas el día 28. Quien ya se retiró no puede recibir ayuda de la herramienta.
- **Objetivo (`riesgo`):** vale 1 si el resultado final es `Fail` o `Withdrawn` (reprobó o abandonó) y 0 si es `Pass` o `Distinction`.

In [3]:
calidad = pd.Series({
    "Duplicados en studentInfo": int(info.duplicated(subset=K).sum()),
    "Duplicados en studentRegistration": int(reg.duplicated(subset=K).sum()),
    "Nulos en imd_band (nivel socioeconómico)": int(info.imd_band.isna().sum()),
    "Notas faltantes en studentAssessment": int(st_assess.score.isna().sum()),
}, name="casos")
display(calidad.to_frame())

base = info.merge(reg, on=K, how="left")
activos = base.date_unregistration.isna() | (base.date_unregistration >= CORTE_DIAS)
base = base[activos].copy()
base["riesgo"] = base.final_result.isin(["Fail", "Withdrawn"]).astype(int)
print(f"Retiradas antes del día {CORTE_DIAS}: {(~activos).sum():,}  |  "
      f"población analizada: {len(base):,} matrículas  |  tasa de riesgo: {base.riesgo.mean():.1%}")

,casos
Duplicados en studentInfo,0
Duplicados en studentRegistration,0
Nulos en imd_band (nivel socioeconómico),1111
Notas faltantes en studentAssessment,173


Retiradas antes del día 28: 5,040  |  población analizada: 27,553 matrículas  |  tasa de riesgo: 44.2%


### 2.2 Indicadores

Cada indicador sigue la definición del diccionario de datos.

| ID | Indicador | Cálculo |
|---|---|---|
| `I1` | Índice de participación semanal (IPS) | clics del estudiante en la semana ÷ mediana de clics de su curso en esa semana |
| `I2` | Días activos | días distintos con al menos un clic entre el día 0 y el 27 |
| `I3` | Entrega pendiente | 1 si alguna evaluación vencida antes del día 28 no se entregó a tiempo |
| `I4` | Nota parcial | promedio (0–100) de lo entregado a tiempo; solo informativa |
| `I6` | Historial y carga | veces que cursó la asignatura y créditos matriculados |
| `I5` | Probabilidad de riesgo | regresión logística sobre `log(1+IPS)`, `I2`, `I3` e `I6` (sección 3) |

In [4]:
SEMANAS = [1, 2, 3, 4]

# Actividad de las primeras cuatro semanas
v = st_vle[(st_vle.date >= 0) & (st_vle.date < CORTE_DIAS)].copy()
v[["code_module", "code_presentation"]] = v[["code_module", "code_presentation"]].astype(str)
v["semana"] = v.date // 7 + 1

# I1: IPS semanal
clics_sem = (v.groupby(K + ["semana"]).sum_click.sum()
               .unstack(fill_value=0).add_prefix("clics_s").reset_index())
df = base.merge(clics_sem, on=K, how="left")
for s in SEMANAS:
    df[f"clics_s{s}"] = df[f"clics_s{s}"].fillna(0)                  # sin registros = 0 clics
    mediana = df.groupby(["code_module", "code_presentation"])[f"clics_s{s}"].transform("median")
    df[f"ips_s{s}"] = (df[f"clics_s{s}"] / mediana.replace(0, np.nan)).fillna(0)
df["ips_prom"] = df[[f"ips_s{s}" for s in SEMANAS]].mean(axis=1)
df["log_ips"] = np.log1p(df.ips_prom)

# I2: días activos
dias = v.groupby(K).date.nunique().rename("dias_activos").reset_index()
df = df.merge(dias, on=K, how="left")
df["dias_activos"] = df.dias_activos.fillna(0).astype(int)

# I3 e I4: entregas y notas hasta el corte (sin el examen final)
ev = assessments[(assessments.assessment_type != "Exam") & (assessments.date < CORTE_DIAS)]
n_venc = ev.groupby(["code_module", "code_presentation"]).size().rename("n_vencidas").reset_index()
ent = st_assess.merge(ev[["id_assessment", "code_module", "code_presentation", "date"]], on="id_assessment")
ent = ent[ent.date_submitted <= ent.date]
ent = ent.groupby(K).agg(n_a_tiempo=("id_assessment", "count"), nota_prom=("score", "mean")).reset_index()
df = df.merge(n_venc, on=["code_module", "code_presentation"], how="left").merge(ent, on=K, how="left")
df[["n_vencidas", "n_a_tiempo"]] = df[["n_vencidas", "n_a_tiempo"]].fillna(0).astype(int)
df["entrega_pendiente"] = ((df.n_vencidas > 0) & (df.n_a_tiempo < df.n_vencidas)).astype(int)

# I6: historial y carga
df["intentos_previos"] = df.num_of_prev_attempts
df["creditos"] = df.studied_credits

NOMBRES = {"ips_prom": "I1 · IPS promedio", "dias_activos": "I2 · Días activos",
           "entrega_pendiente": "I3 · Entrega pendiente", "nota_prom": "I4 · Nota parcial",
           "intentos_previos": "I6 · Veces cursada", "creditos": "I6 · Créditos"}
display(df[list(NOMBRES)].rename(columns=NOMBRES).describe().T
          [["count", "mean", "std", "min", "50%", "max"]].round(2))

,count,mean,std,min,50%,max
I1 · IPS promedio,27553.0,2.06,3.36,0.0,1.25,95.89
I2 · Días activos,27553.0,10.65,7.19,0.0,10.00,28.00
I3 · Entrega pendiente,27553.0,0.28,0.45,0.0,0.00,1.00
I4 · Nota parcial,15831.0,73.83,20.90,0.0,78.00,100.00
I6 · Veces cursada,27553.0,0.16,0.47,0.0,0.00,6.00
I6 · Créditos,27553.0,76.37,38.28,30.0,60.00,630.00


### 2.3 ¿Los indicadores distinguen a quienes terminan en riesgo?

Hacemos una primera validación descriptiva: si `I1`–`I3` fueran iguales entre quienes aprueban y quienes no, no servirían como señal temprana.

In [5]:
orden_res = ["Distinction", "Pass", "Fail", "Withdrawn"]
etiquetas_res = {"Distinction": "Distinción", "Pass": "Aprobó", "Fail": "Reprobó", "Withdrawn": "Se retiró"}
tabla_res = (df.groupby("final_result")[["ips_prom", "dias_activos", "entrega_pendiente", "nota_prom"]]
               .mean().loc[orden_res].rename(index=etiquetas_res, columns=NOMBRES))
display(tabla_res.round(2))

colores_res = ["#1f77b4", COLOR["Verde"], COLOR["Rojo"], "#8c564b"]
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                    subplot_titles=("I1 · IPS promedio (semanas 1–4)", "I2 · Días activos (de 28)"))
for res, col in zip(orden_res, colores_res):
    d = df[df.final_result == res]
    fig.add_trace(go.Box(y=d.ips_prom.clip(upper=6), name=etiquetas_res[res], marker_color=col,
                         boxpoints=False, showlegend=False), row=1, col=1)
    fig.add_trace(go.Box(y=d.dias_activos, name=etiquetas_res[res], marker_color=col,
                         boxpoints=False, showlegend=False), row=1, col=2)
fig.update_yaxes(title_text="IPS (1 = mediana del curso)", row=1, col=1)
fig.update_yaxes(title_text="días", row=1, col=2)
fig.update_layout(height=440, font=FUENTE, margin=dict(t=95, b=40, l=60, r=20),
                  title=dict(text="Los estudiantes que reprueban o se retiran participan menos desde el inicio", x=0.01))
fig.show()

,I1 · IPS promedio,I2 · Días activos,I3 · Entrega pendiente,I4 · Nota parcial
final_result,,,,
Distinción,3.42,14.20,0.20,82.68
Aprobó,2.38,12.08,0.18,75.35
Reprobó,1.35,7.63,0.35,69.39
Se retiró,1.47,9.28,0.46,68.80


## 3. Modelo base de riesgo (`I5`)

- **Partición temporal:** entrenamos con los períodos de **2013** y evaluamos con los de **2014**. Así imitamos el uso real: se entrena con cohortes pasadas y se predice la actual. Una partición aleatoria daría resultados demasiado optimistas.
- **Modelos:** una regresión logística, que es el modelo de la herramienta y es fácil de explicar al tutor, y un árbol de decisión como comparación.
- **Variables excluidas:** las demográficas (sexo, discapacidad, edad, nivel socioeconómico) no entran al modelo por razones éticas. Solo se usan para revisar sesgos en la sección 5.

In [6]:
FEATURES = ["log_ips", "dias_activos", "entrega_pendiente", "intentos_previos", "creditos"]
train = df[df.code_presentation.str.startswith("2013")]
test = df[df.code_presentation.str.startswith("2014")].copy()

logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
logit.fit(train[FEATURES], train.riesgo)
arbol = DecisionTreeClassifier(max_depth=4, min_samples_leaf=100, class_weight="balanced", random_state=SEMILLA)
arbol.fit(train[FEATURES], train.riesgo)

test["p_riesgo"] = logit.predict_proba(test[FEATURES])[:, 1]
p_arbol = arbol.predict_proba(test[FEATURES])[:, 1]
test["semaforo"] = np.select([test.p_riesgo >= UMBRAL_ROJO, test.p_riesgo >= UMBRAL_AMARILLO],
                             ["Rojo", "Amarillo"], "Verde")

aucs = pd.Series({"Regresión logística (modelo de la herramienta)": roc_auc_score(test.riesgo, test.p_riesgo),
                  "Árbol de decisión (comparación)": roc_auc_score(test.riesgo, p_arbol),
                  "Solo IPS, sin modelo (referencia)": roc_auc_score(test.riesgo, -test.ips_prom)}, name="AUC")
print(f"Entrenamiento (2013): {len(train):,}  |  prueba (2014): {len(test):,}  |  "
      f"tasa de riesgo en prueba: {test.riesgo.mean():.1%}")
display(aucs.round(3).to_frame())

Entrenamiento (2013): 11,818  |  prueba (2014): 15,735  |  tasa de riesgo en prueba: 44.0%


,AUC
Regresión logística (modelo de la herramienta),0.716
Árbol de decisión (comparación),0.702
"Solo IPS, sin modelo (referencia)",0.691


In [7]:
filas = []
for u in [0.3, 0.4, 0.5, 0.6, 0.7]:
    pred = (test.p_riesgo >= u).astype(int)
    filas.append({"umbral": u, "recall": recall_score(test.riesgo, pred),
                  "precisión": precision_score(test.riesgo, pred), "% del curso alertado": pred.mean()})
tabla_umbral = pd.DataFrame(filas).set_index("umbral")

fpr, tpr, _ = roc_curve(test.riesgo, test.p_riesgo)
fpr_a, tpr_a, _ = roc_curve(test.riesgo, p_arbol)
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                    subplot_titles=("Curva ROC (cohorte de prueba 2014)", "Recall y precisión según el umbral"))
fig.add_trace(go.Scatter(x=fpr, y=tpr, name=f"Logística (AUC {dec(aucs.iloc[0])})",
                         line=dict(color=COLOR["azul"], width=3)), row=1, col=1)
fig.add_trace(go.Scatter(x=fpr_a, y=tpr_a, name=f"Árbol (AUC {dec(aucs.iloc[1])})",
                         line=dict(color=COLOR["gris"], width=2, dash="dot")), row=1, col=1)
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], showlegend=False, line=dict(color="#bbbbbb", dash="dash")), row=1, col=1)
fig.add_trace(go.Scatter(x=tabla_umbral.index, y=tabla_umbral.recall, name="Recall (detectados)",
                         mode="lines+markers", line=dict(color=COLOR["Rojo"], width=3)), row=1, col=2)
fig.add_trace(go.Scatter(x=tabla_umbral.index, y=tabla_umbral["precisión"], name="Precisión (alertas correctas)",
                         mode="lines+markers", line=dict(color=COLOR["azul"], width=3)), row=1, col=2)
fig.add_trace(go.Scatter(x=tabla_umbral.index, y=tabla_umbral["% del curso alertado"], name="% del curso alertado",
                         mode="lines", line=dict(color=COLOR["gris"], dash="dash")), row=1, col=2)
for u, nombre in [(UMBRAL_AMARILLO, "Amarillo"), (UMBRAL_ROJO, "Rojo")]:
    fig.add_vline(x=u, line=dict(color=COLOR[nombre], width=8), opacity=0.25, row=1, col=2)
    fig.add_annotation(x=u, y=1.04, text=nombre, showarrow=False, font=dict(color=COLOR[nombre], size=12),
                       row=1, col=2)
fig.update_xaxes(title_text="tasa de falsas alarmas", row=1, col=1)
fig.update_yaxes(title_text="recall", row=1, col=1)
fig.update_xaxes(title_text="umbral de probabilidad", row=1, col=2)
fig.update_yaxes(range=[0, 1.1], row=1, col=2)
fig.update_layout(height=460, font=FUENTE, margin=dict(t=70, b=110, l=60, r=20),
                  legend=dict(orientation="h", y=-0.22, x=0.5, xanchor="center"))
fig.show()
display(tabla_umbral.style.format("{:.2f}"))

,recall,precisión,% del curso alertado
umbral,,,
0.300000,0.92,0.49,0.84
0.400000,0.82,0.54,0.67
0.500000,0.66,0.59,0.49
0.600000,0.50,0.67,0.33
0.700000,0.34,0.74,0.20


In [8]:
alerta = test.p_riesgo >= UMBRAL_AMARILLO
rojo = test.p_riesgo >= UMBRAL_ROJO
tp = (alerta & (test.riesgo == 1)).sum(); fn = (~alerta & (test.riesgo == 1)).sum(); fp = (alerta & (test.riesgo == 0)).sum()
print("Lectura para el tutor")
print(f"• Con alerta amarilla o roja (p ≥ {dec(UMBRAL_AMARILLO)}): de cada 100 estudiantes que terminan reprobando o "
      f"retirándose, la herramienta alerta a {100*tp/(tp+fn):.0f} en la semana 4 y no detecta a {100*fn/(tp+fn):.0f}.")
print(f"• De cada 100 alertas, {100*tp/(tp+fp):.0f} son estudiantes que realmente terminan en riesgo.")
print(f"• Solo rojo (p ≥ {dec(UMBRAL_ROJO)}): precisión {dec(precision_score(test.riesgo, rojo))}, "
      f"recall {dec(recall_score(test.riesgo, rojo))}, {rojo.mean():.0%} del curso.")

Lectura para el tutor
• Con alerta amarilla o roja (p ≥ 0,40): de cada 100 estudiantes que terminan reprobando o retirándose, la herramienta alerta a 82 en la semana 4 y no detecta a 18.
• De cada 100 alertas, 54 son estudiantes que realmente terminan en riesgo.
• Solo rojo (p ≥ 0,60): precisión 0,67, recall 0,50, 33% del curso.


**Qué significan estas métricas para el tutor**

- **AUC ≈ 0,72:** si se toma al azar un estudiante que terminará en riesgo y otro que no, el modelo le asigna mayor probabilidad al primero unas 7 de cada 10 veces. Es una capacidad **moderada**: sirve para **priorizar** a quién llamar primero, no para etiquetar a nadie.
- **Recall:** proporción de estudiantes en riesgo que la alerta detecta. Para el tutor es la métrica más importante, porque no detectar a alguien que luego abandona cuesta más que una conversación innecesaria.
- **Precisión:** proporción de alertas correctas. Si es baja, el tutor dedica parte de su tiempo a estudiantes que no lo necesitaban.
- **Por eso el semáforo tiene dos niveles.** El amarillo activa una acción barata (un mensaje con recursos) y tolera falsas alarmas. El rojo activa una acción costosa (una tutoría individual) y se reserva para las probabilidades más altas.

In [9]:
coef = pd.Series(logit[-1].coef_[0], index=FEATURES)
nombres_modelo = {"log_ips": "I1 · IPS (log)", "dias_activos": "I2 · Días activos", "entrega_pendiente": "I3 · Entrega pendiente",
                  "intentos_previos": "I6 · Veces cursada", "creditos": "I6 · Créditos matriculados"}
coef_df = pd.DataFrame({"coeficiente estandarizado": coef, "odds ratio por 1 DE": np.exp(coef)})
display(coef_df.rename(index=nombres_modelo).sort_values("coeficiente estandarizado").round(2))

por_curso = test.groupby(["code_module", "code_presentation"]).agg(
    estudiantes=("riesgo", "size"), tasa_real_de_riesgo=("riesgo", "mean"),
    porcentaje_en_rojo=("semaforo", lambda s: (s == "Rojo").mean()),
    auc=("p_riesgo", lambda p: roc_auc_score(test.loc[p.index, "riesgo"], p)))
display(por_curso.style.format({"tasa_real_de_riesgo": "{:.0%}", "porcentaje_en_rojo": "{:.0%}", "auc": "{:.2f}"}))
print("Con un mismo umbral, el porcentaje de estudiantes en rojo cambia mucho de un curso a otro: "
      "habría que calibrar por asignatura (ver sección 6).")

,coeficiente estandarizado,odds ratio por 1 DE
I2 · Días activos,-0.40,0.67
I1 · IPS (log),-0.27,0.76
I6 · Veces cursada,0.16,1.17
I6 · Créditos matriculados,0.17,1.18
I3 · Entrega pendiente,0.47,1.61


Con un mismo umbral, el porcentaje de estudiantes en rojo cambia mucho de un curso a otro: habría que calibrar por asignatura (ver sección 6).


## 4. Prototipo de la herramienta: pantallas P1 a P4

Construimos las cuatro pantallas de los wireframes con los datos de un curso real de la cohorte de prueba, **AAA-2014J**, tal como las vería un tutor al cerrar la semana 4. Respetamos la misma distribución, los mismos elementos y los mismos indicadores.

OULAD no tiene registro de intervenciones ni de consentimiento, porque esos datos los generaría la propia herramienta. En las pantallas aparecen como **0 registros**: es el estado real de la herramienta en la primera semana de uso.

Las funciones siguientes dibujan los elementos comunes: barra superior, paneles, tarjetas y botones.

In [10]:
def lienzo(alto=1000):
    fig = go.Figure()
    fig.update_layout(height=alto, width=1250, font=FUENTE, plot_bgcolor="white", paper_bgcolor="white",
                      margin=dict(l=10, r=10, t=10, b=10), showlegend=False)
    return fig

def rect(fig, x0, y0, x1, y1, relleno="white", borde=None, ancho=1.2):
    fig.add_shape(type="rect", xref="paper", yref="paper", x0=x0, y0=y0, x1=x1, y1=y1, layer="below",
                  fillcolor=relleno, line=dict(color=borde or COLOR["borde"], width=ancho))

def texto(fig, x, y, t, size=13, color="#1f2933", bold=False, xanchor="left", yanchor="top", align="left"):
    fig.add_annotation(xref="paper", yref="paper", x=x, y=y, text=f"<b>{t}</b>" if bold else t, showarrow=False,
                       font=dict(size=size, color=color), xanchor=xanchor, yanchor=yanchor, align=align)

def barra_superior(fig, pantalla, actor, subtitulo):
    rect(fig, 0, 0.955, 1, 1, relleno="#1f4e79", borde="#1f4e79")
    texto(fig, 0.012, 0.99, "Radar Tutor UC", 18, "white", bold=True)
    texto(fig, 0.17, 0.988, pantalla, 15, "white")
    texto(fig, 0.988, 0.988, actor, 13, "#dfe9f5", xanchor="right")
    rect(fig, 0, 0.915, 1, 0.948, relleno=COLOR["fondo"])
    texto(fig, 0.012, 0.941, subtitulo, 12, "#4a5568")

def boton(fig, x0, y0, ancho, etiqueta, relleno="white", borde=None, color="#1f2933"):
    rect(fig, x0, y0, x0 + ancho, y0 + 0.032, relleno=relleno, borde=borde or "#9aa5b1")
    texto(fig, x0 + ancho / 2, y0 + 0.016, etiqueta, 12, color, xanchor="center", yanchor="middle")

def kpi(fig, x0, x1, y0, y1, titulo, valor, color="#1f2933", nota=""):
    rect(fig, x0, y0, x1, y1, borde=COLOR["borde"])
    rect(fig, x0, y1 - 0.006, x1, y1, relleno=color, borde=color)
    texto(fig, (x0 + x1) / 2, y1 - 0.016, titulo, 13, "#4a5568", xanchor="center")
    texto(fig, (x0 + x1) / 2, (y0 + y1) / 2 - 0.008, f"<b>{valor}</b>", 30, color, xanchor="center", yanchor="middle")
    if nota:
        texto(fig, (x0 + x1) / 2, y0 + 0.012, nota, 10, "#7b8794", xanchor="center", yanchor="bottom")

def ejes(fig, n, x, y, **kw):
    """Coloca el eje n (1, 2, 3…) en el dominio indicado del lienzo."""
    sx = "" if n == 1 else str(n)
    fig.update_layout(**{f"xaxis{sx}": dict(domain=x, anchor=f"y{sx}", **kw.get("xa", {})),
                         f"yaxis{sx}": dict(domain=y, anchor=f"x{sx}", **kw.get("ya", {}))})
    return f"x{sx}", f"y{sx}"

# Curso de demostración (cohorte de prueba)
CURSO = ("AAA", "2014J")
c = test[(test.code_module == CURSO[0]) & (test.code_presentation == CURSO[1])].copy()
dur_semanas = int(np.ceil(courses.set_index(["code_module", "code_presentation"])
                          .loc[CURSO, "module_presentation_length"] / 7))
c["accion"] = c.semaforo.map({"Rojo": "Agendar tutoría", "Amarillo": "Enviar mensaje", "Verde": "—"})
ips_cols = [f"ips_s{s}" for s in SEMANAS]
print(f"Curso {CURSO[0]}-{CURSO[1]}: {len(c)} estudiantes activos en la semana 4 de {dur_semanas}.")

Curso AAA-2014J: 348 estudiantes activos en la semana 4 de 39.


### P1 · Vista del curso (tutor)

Esta es la pantalla principal. Muestra: ① las tarjetas del curso, ② el semáforo `I5`, ③ la participación semanal `I1` por grupo, ④ la lista priorizada de estudiantes con sus indicadores, ⑤–⑥ las acciones disponibles y ⑦ la cobertura `I7`.

In [11]:
def pantalla_p1(c):
    fig = lienzo(1080)
    barra_superior(fig, "P1 · Vista del curso", "Tutor académico",
                   f"Curso: <b>{CURSO[0]} · {CURSO[1]}</b> ▾      Semana: <b>4 de {dur_semanas}</b> ▾      "
                   "Última actualización: lunes 06:00")
    n = c.semaforo.value_counts().reindex(["Rojo", "Amarillo", "Verde"], fill_value=0)
    kp = [("Estudiantes activos", len(c), COLOR["azul"], "matrícula vigente"),
          ("En rojo", n["Rojo"], COLOR["Rojo"], "tutoría en ≤ 7 días"),
          ("En amarillo", n["Amarillo"], COLOR["Amarillo"], "mensaje en ≤ 3 días"),
          ("IPS mediano (I1)", dec(c.ips_prom.median()), COLOR["azul"], "1 = mediana del curso")]
    for i, (t, val, col, nota) in enumerate(kp):
        x0 = i * 0.25 + (0 if i == 0 else 0.008)
        kpi(fig, x0, x0 + 0.242, 0.775, 0.895, t, val, col, nota)

    # ② semáforo
    rect(fig, 0, 0.445, 0.475, 0.755)
    texto(fig, 0.012, 0.748, "Distribución del semáforo de riesgo (I5)", 14, bold=True)
    xa, ya = ejes(fig, 1, [0.04, 0.455], [0.475, 0.70], ya=dict(title="estudiantes", gridcolor="#eef1f5"))
    fig.add_trace(go.Bar(x=n.index, y=n.values, marker_color=[COLOR[k] for k in n.index], text=n.values,
                         textposition="outside", cliponaxis=False, xaxis=xa, yaxis=ya, width=0.55))

    # ③ participación semanal
    rect(fig, 0.49, 0.445, 1, 0.755)
    texto(fig, 0.502, 0.748, "Participación semanal (I1)", 14, bold=True)
    xa, ya = ejes(fig, 2, [0.535, 0.84], [0.475, 0.70],
                  xa=dict(tickvals=SEMANAS, ticktext=[f"S{s}" for s in SEMANAS]),
                  ya=dict(title="IPS", gridcolor="#eef1f5", rangemode="tozero"))
    fig.add_trace(go.Scatter(x=SEMANAS, y=[1] * 4, mode="lines", line=dict(color=COLOR["gris"], dash="dash"),
                             xaxis=xa, yaxis=ya))
    for grupo in ["Verde", "Rojo"]:
        fig.add_trace(go.Scatter(x=SEMANAS, y=c[c.semaforo == grupo][ips_cols].median().values,
                                 mode="lines+markers", line=dict(color=COLOR[grupo], width=3), marker=dict(size=8),
                                 xaxis=xa, yaxis=ya))
    for y, t, col in [(0.66, "— grupo verde", COLOR["Verde"]), (0.62, "— grupo rojo", COLOR["Rojo"]),
                      (0.58, "- - mediana del curso", COLOR["gris"])]:
        texto(fig, 0.855, y, t, 12, col)

    # ④ lista priorizada
    texto(fig, 0.0, 0.43, "Estudiantes priorizados (ordenados por riesgo)", 14, bold=True)
    top = c.sort_values("p_riesgo", ascending=False).head(12)
    fondo = {"Rojo": "#fbe5e5", "Amarillo": "#fff4cc", "Verde": "#e6f4e6"}
    fig.add_trace(go.Table(
        domain=dict(x=[0, 1], y=[0.085, 0.405]),
        columnwidth=[1.3, 0.8, 0.8, 0.7, 0.8, 1.1, 0.9, 1.1, 1.3],
        header=dict(values=["<b>Estudiante</b>", "<b>Riesgo (I5)</b>", "<b>Semáforo</b>", "<b>IPS (I1)</b>",
                            "<b>Días act. (I2)</b>", "<b>Entrega pend. (I3)</b>", "<b>Nota parcial (I4)</b>",
                            "<b>Última interv. (I7)</b>", "<b>Acción</b>"],
                    fill_color="#e9edf3", align="left", font=dict(size=12), height=30),
        cells=dict(values=[[f"Est. {i}" for i in top.id_student], top.p_riesgo.map(dec), top.semaforo,
                           top.ips_prom.map(dec), top.dias_activos,
                           top.entrega_pendiente.map({1: "Sí", 0: "No"}),
                           top.nota_prom.map(lambda x: "—" if pd.isna(x) else f"{x:.0f}"), ["—"] * len(top),
                           ["[Ficha] [" + ("Agendar" if s == "Rojo" else "Mensaje") + "]" for s in top.semaforo]],
                   fill_color=[[fondo[s] for s in top.semaforo]], align="left", height=25, font=dict(size=12))))

    # ⑤ ⑥ acciones y ⑦ cobertura
    boton(fig, 0.0, 0.043, 0.30, "✉  Enviar mensaje a todos los amarillos", relleno="#fff4cc", borde=COLOR["Amarillo"])
    boton(fig, 0.32, 0.043, 0.22, "⬇  Exportar lista (sin notas)")
    boton(fig, 0.56, 0.043, 0.25, "?  ¿Cómo se calcula el riesgo?")
    rect(fig, 0, 0.0, 1, 0.034, relleno="#e8f1fb", borde="#b9d3ee")
    texto(fig, 0.5, 0.017, f"Cobertura (I7): 0 de {n['Rojo']} estudiantes en rojo con una intervención registrada "
                          "(primera semana de uso de la herramienta)", 12, "#1f4e79", xanchor="center", yanchor="middle")
    return fig

fig_p1 = pantalla_p1(c)
fig_p1.write_html("tablero_P1.html", include_plotlyjs="cdn")
fig_p1.show()

### P2 · Ficha del estudiante (tutor)

Se abre desde la lista de P1. Mostramos al estudiante con mayor riesgo del curso: ① el medidor `I5`, ② su IPS semanal frente al curso, ③ el calendario de días activos, ④ sus entregas y notas, ⑤ el historial y la carga, ⑥ la explicación del modelo, ⑦ las acciones y ⑧ el historial de intervenciones.

In [12]:
def contribuciones(fila):
    z = logit[0].transform(fila[FEATURES].to_frame().T.astype(float))[0]
    return pd.Series(logit[-1].coef_[0] * z, index=[nombres_modelo[f] for f in FEATURES]).sort_values()

def entregas_de(est):
    a = assessments[(assessments.code_module == CURSO[0]) & (assessments.code_presentation == CURSO[1]) &
                    (assessments.assessment_type != "Exam")].sort_values("date")
    e = st_assess[st_assess.id_student == est.id_student].set_index("id_assessment")
    a["nombre"] = [f"Tarea {i}" if t == "TMA" else f"Cuestionario {i}"
                   for i, t in zip(a.groupby("assessment_type").cumcount() + 1, a.assessment_type)]
    lineas = []
    for _, r in a.iterrows():
        if r.date < CORTE_DIAS:
            if r.id_assessment in e.index and e.loc[r.id_assessment, "date_submitted"] <= r.date:
                lineas.append((f"✓ {r.nombre} · venció el día {r.date:.0f} · entregada · "
                               f"{e.loc[r.id_assessment, 'score']:.0f}/100", COLOR["Verde"]))
            else:
                lineas.append((f"✗ {r.nombre} · venció el día {r.date:.0f} · no entregada",
                               COLOR["Rojo"]))
        elif len(lineas) < 4:
            lineas.append((f"◷ {r.nombre} · vence el día {r.date:.0f}", "#4a5568"))
    return lineas

def pantalla_p2(est):
    fig = lienzo(1060)
    barra_superior(fig, "P2 · Ficha del estudiante", "Tutor académico",
                   "← Volver al curso      " f"Curso {CURSO[0]} · {CURSO[1]} · semana 4")
    # ⑤ datos del estudiante
    rect(fig, 0, 0.70, 0.33, 0.895)
    texto(fig, 0.012, 0.885, f"Estudiante {est.id_student}", 16, bold=True)
    texto(fig, 0.012, 0.845, "(seudónimo; el nombre real solo lo ve su tutor)", 11, "#7b8794")
    texto(fig, 0.012, 0.81, f"Veces que cursó la asignatura (I6): <b>{int(est.intentos_previos)}</b>", 13)
    texto(fig, 0.012, 0.775, f"Créditos matriculados (I6): <b>{int(est.creditos)}</b>", 13)
    texto(fig, 0.012, 0.74, "Consentimiento: <b>vigente</b>", 13, COLOR["Verde"])
    # ① medidor de riesgo
    rect(fig, 0.34, 0.70, 0.66, 0.895)
    texto(fig, 0.352, 0.885, "Probabilidad de riesgo (I5)", 14, bold=True)
    fig.add_trace(go.Indicator(mode="gauge+number", value=est.p_riesgo, number=dict(valueformat=".2f", font=dict(size=34)),
                               domain=dict(x=[0.38, 0.62], y=[0.705, 0.845]),
                               gauge=dict(axis=dict(range=[0, 1], tickvals=[0, 0.4, 0.6, 1]),
                                          bar=dict(color="#1f2933", thickness=0.25),
                                          steps=[dict(range=[0, 0.4], color="#cfe8cf"), dict(range=[0.4, 0.6], color="#fbe7a6"),
                                                 dict(range=[0.6, 1], color="#f4b6b6")])))
    # ④ entregas y notas
    rect(fig, 0.67, 0.70, 1, 0.895)
    texto(fig, 0.682, 0.885, "Entregas (I3) y notas (I4)", 14, bold=True)
    for i, (t, col) in enumerate(entregas_de(est)[:5]):
        texto(fig, 0.682, 0.845 - i * 0.03, t, 12, col)

    # ② IPS semanal
    rect(fig, 0, 0.385, 0.49, 0.685)
    texto(fig, 0.012, 0.678, "IPS semanal del estudiante frente al curso (I1)", 14, bold=True)
    xa, ya = ejes(fig, 1, [0.05, 0.47], [0.42, 0.63], ya=dict(title="IPS", gridcolor="#eef1f5", rangemode="tozero"))
    fig.add_trace(go.Bar(x=[f"S{s}" for s in SEMANAS], y=[est[f"ips_s{s}"] for s in SEMANAS], marker_color=COLOR["Rojo"],
                         text=[dec(est[f"ips_s{s}"]) for s in SEMANAS], textposition="outside", cliponaxis=False,
                         xaxis=xa, yaxis=ya, width=0.5))
    fig.add_trace(go.Scatter(x=[f"S{s}" for s in SEMANAS], y=[1] * 4, mode="lines", line=dict(color=COLOR["gris"], dash="dash"),
                             xaxis=xa, yaxis=ya))
    texto(fig, 0.47, 0.645, "- - mediana del curso (IPS = 1)", 11, COLOR["gris"], xanchor="right")

    # ③ calendario de días activos
    rect(fig, 0.50, 0.385, 1, 0.685)
    texto(fig, 0.512, 0.678, "Días activos en eVirtual (I2)", 14, bold=True)
    dias_est = set(v[(v.id_student == est.id_student) & (v.code_module == CURSO[0]) &
                     (v.code_presentation == CURSO[1])].date)
    z = [[1 if (s - 1) * 7 + d in dias_est else 0 for d in range(7)] for s in SEMANAS]
    xa, ya = ejes(fig, 2, [0.56, 0.97], [0.44, 0.635], xa=dict(showgrid=False),
                  ya=dict(autorange="reversed", showgrid=False))
    fig.add_trace(go.Heatmap(z=z, x=[f"día {d+1}" for d in range(7)], y=[f"S{s}" for s in SEMANAS],
                             colorscale=[[0, "#e9edf3"], [1, COLOR["azul"]]], showscale=False, xgap=4, ygap=4,
                             zmin=0, zmax=1, xaxis=xa, yaxis=ya))
    texto(fig, 0.512, 0.405, f"{int(est.dias_activos)} día(s) activo(s) de 28 · se cuentan días, no horarios", 12, "#4a5568")

    # ⑥ explicación del modelo
    rect(fig, 0, 0.075, 0.49, 0.37)
    texto(fig, 0.012, 0.363, "¿Por qué está en alerta? (aporte de cada indicador al riesgo)", 14, bold=True)
    contrib = contribuciones(est)
    texto(fig, 0.012, 0.335, "rojo = sube el riesgo · verde = lo baja", 11, "#7b8794")
    xa, ya = ejes(fig, 3, [0.20, 0.47], [0.11, 0.31], xa=dict(zeroline=True, zerolinecolor="#9aa5b1"),
                  ya=dict(automargin=True))
    fig.add_trace(go.Bar(x=contrib.values, y=contrib.index, orientation="h", xaxis=xa, yaxis=ya,
                         marker_color=[COLOR["Rojo"] if x > 0 else COLOR["Verde"] for x in contrib.values]))
    # ⑧ historial de intervenciones
    rect(fig, 0.50, 0.075, 1, 0.37)
    texto(fig, 0.512, 0.363, "Historial de intervenciones (I7)", 14, bold=True)
    fig.add_trace(go.Table(domain=dict(x=[0.512, 0.988], y=[0.17, 0.335]),
                           header=dict(values=["<b>Fecha</b>", "<b>Acción</b>", "<b>Por</b>", "<b>Resultado</b>"],
                                       fill_color="#e9edf3", align="left"),
                           cells=dict(values=[["—"], ["Sin intervenciones registradas todavía"], ["—"], ["—"]],
                                      align="left", height=26)))
    # ⑦ acciones y aviso
    boton(fig, 0.0, 0.03, 0.22, "📅  Agendar tutoría", relleno="#fbe5e5", borde=COLOR["Rojo"])
    boton(fig, 0.235, 0.03, 0.22, "✉  Enviar mensaje")
    boton(fig, 0.47, 0.03, 0.28, "→  Derivar a Bienestar Universitario")
    texto(fig, 0.0, 0.012, "El riesgo es una estimación para priorizar la conversación; no es un diagnóstico "
                           "y no se comparte con los docentes de la asignatura.", 11, "#7b6a00", yanchor="middle")
    return fig

est = c.sort_values("p_riesgo", ascending=False).iloc[0]
pantalla_p2(est).show()

### P3 · Registro de intervención (tutor)

Es el formulario que se abre desde *Agendar tutoría* o *Enviar mensaje*. El motivo se llena solo con los indicadores del estudiante. A la derecha están la cola de casos en rojo de la semana y el seguimiento de la cobertura `I7`.

In [13]:
def pantalla_p3(est, c):
    fig = lienzo(1000)
    barra_superior(fig, "P3 · Registro de intervención", "Tutor académico", "← Volver a la ficha")
    rect(fig, 0, 0.17, 0.60, 0.895)
    texto(fig, 0.012, 0.885, f"Nueva intervención · Est. {est.id_student} (riesgo {dec(est.p_riesgo)} · {est.semaforo})", 16, bold=True)
    motivo = (f"IPS {dec(est.ips_prom)} · {int(est.dias_activos)} día(s) activo(s) · "
              f"{'entrega pendiente' if est.entrega_pendiente else 'sin entregas pendientes'}")
    campos = [("Motivo detectado (automático)", motivo),
              ("Tipo de acción", "Tutoría individual ▾      (mensaje · tutoría · derivación a Bienestar)"),
              ("Canal y fecha", "Presencial ▾      lunes de la semana 5 · 10:00"),
              ("Acuerdos y compromisos", "p. ej., ponerse al día con la evaluación vencida; plan de estudio semanal"),
              ("Fecha de seguimiento", "lunes de la semana 6"),
              ("Resultado", "○ Asistió     ○ No asistió     ○ Reprogramada")]
    for i, (et, val) in enumerate(campos):
        y = 0.84 - i * 0.105
        texto(fig, 0.015, y, et, 13, bold=True)
        rect(fig, 0.015, y - 0.075, 0.585, y - 0.03, relleno="#fafbfc", borde="#c3cbd6")
        texto(fig, 0.025, y - 0.052, val, 12, "#4a5568", yanchor="middle")
    boton(fig, 0.015, 0.19, 0.2, "💾  Guardar registro", relleno="#e6f4e6", borde=COLOR["Verde"])
    boton(fig, 0.23, 0.19, 0.15, "Cancelar")

    rect(fig, 0.615, 0.42, 1, 0.895)
    texto(fig, 0.627, 0.885, "Intervenciones de esta semana", 14, bold=True)
    cola = c[c.semaforo == "Rojo"].sort_values("p_riesgo", ascending=False).head(10)
    fig.add_trace(go.Table(domain=dict(x=[0.627, 0.988], y=[0.44, 0.85]),
                           header=dict(values=["<b>Estudiante</b>", "<b>Riesgo</b>", "<b>Acción</b>", "<b>Estado</b>"],
                                       fill_color="#e9edf3", align="left"),
                           cells=dict(values=[[f"Est. {i}" for i in cola.id_student], cola.p_riesgo.map(dec),
                                              ["Tutoría"] * len(cola), ["Pendiente"] * len(cola)],
                                      fill_color="#fbe5e5", align="left", height=24)))
    rect(fig, 0.615, 0.17, 1, 0.405, relleno="#e8f1fb", borde="#b9d3ee")
    n_rojo = (c.semaforo == "Rojo").sum()
    texto(fig, 0.627, 0.395, "Seguimiento (I7)", 14, "#1f4e79", bold=True)
    for i, t in enumerate([f"Rojos con acción en ≤ 7 días: 0 de {n_rojo} (0 %)", "Tutorías realizadas: 0",
                           f"Mensajes enviados a amarillos: 0 de {(c.semaforo == 'Amarillo').sum()}",
                           "Estos registros alimentan el plan de validación."]):
        texto(fig, 0.627, 0.345 - i * 0.04, t, 13, "#1f2933" if i < 3 else "#7b8794")

    rect(fig, 0, 0.0, 1, 0.155)
    texto(fig, 0.012, 0.145, "Plantilla de mensaje (amarillos) · editable antes de enviar", 14, bold=True)
    texto(fig, 0.012, 0.10, f"«Hola, notamos que en las últimas semanas tu participación en {CURSO[0]} bajó. "
                            "Te compartimos estos recursos para ponerte al día: [enlaces].", 12, "#4a5568")
    texto(fig, 0.012, 0.065, "Si necesitas apoyo, puedes agendar una tutoría conmigo aquí: [enlace]. "
                             "¡Estamos para ayudarte!»", 12, "#4a5568")
    return fig

pantalla_p3(est, c).show()

### P4 · Mi progreso (estudiante)

Es la vista del estudiante. Por decisión ética **no muestra la probabilidad de riesgo ni un ranking**: solo los indicadores propios del estudiante comparados con la mediana del curso, sus entregas y acciones concretas. Usamos como ejemplo un estudiante en amarillo.

In [14]:
def pantalla_p4(est):
    fig = lienzo(940)
    barra_superior(fig, "P4 · Mi progreso", "Estudiante", f"Asignatura {CURSO[0]} · semana 4 de {dur_semanas}")
    rect(fig, 0, 0.83, 1, 0.905, relleno="#e6f4e6", borde="#b7dfb7")
    texto(fig, 0.012, 0.893, f"Hola 👋  Esta es tu actividad en {CURSO[0]} hasta la semana 4.", 15, bold=True)
    texto(fig, 0.012, 0.855, "Solo tú y tu tutor ven esta información ·  <u>¿Qué datos se usan?</u>  ·  "
                             "<u>Gestionar mi consentimiento</u>", 12, "#1f4e79")

    rect(fig, 0, 0.46, 0.49, 0.815)
    texto(fig, 0.012, 0.808, "Tu participación semanal comparada con la del curso (I1)", 14, bold=True)
    xa, ya = ejes(fig, 1, [0.05, 0.47], [0.50, 0.75], ya=dict(title="IPS", gridcolor="#eef1f5", rangemode="tozero"))
    fig.add_trace(go.Bar(x=[f"S{s}" for s in SEMANAS], y=[est[f"ips_s{s}"] for s in SEMANAS], marker_color=COLOR["azul"],
                         xaxis=xa, yaxis=ya, width=0.5))
    fig.add_trace(go.Scatter(x=[f"S{s}" for s in SEMANAS], y=[1] * 4, mode="lines", line=dict(color=COLOR["gris"], dash="dash"),
                             xaxis=xa, yaxis=ya))
    texto(fig, 0.47, 0.775, "- - mediana del curso", 11, COLOR["gris"], xanchor="right")

    rect(fig, 0.50, 0.46, 1, 0.815)
    texto(fig, 0.512, 0.808, "Tus días activos (I2)", 14, bold=True)
    texto(fig, 0.75, 0.68, f"<b>{int(est.dias_activos)} de 28 días</b>", 40, COLOR["azul"], xanchor="center", yanchor="middle")
    texto(fig, 0.75, 0.56, "Consejo: estudiar un poco varios días a la semana<br>funciona mejor que hacerlo todo en un solo día.",
          13, "#4a5568", xanchor="center", yanchor="middle", align="center")

    rect(fig, 0, 0.15, 0.49, 0.445)
    texto(fig, 0.012, 0.438, "Tus entregas (I3) y notas (I4)", 14, bold=True)
    for i, (t, col) in enumerate(entregas_de(est)[:5]):
        texto(fig, 0.012, 0.395 - i * 0.045, t, 13, col)

    rect(fig, 0.50, 0.15, 1, 0.445)
    texto(fig, 0.512, 0.438, "Recursos sugeridos y apoyo", 14, bold=True)
    texto(fig, 0.512, 0.395, "• Video: repaso del tema de la semana 3", 13)
    texto(fig, 0.512, 0.35, "• Guía de ejercicios resueltos", 13)
    boton(fig, 0.512, 0.20, 0.2, "📅  Solicitar tutoría", relleno="#fbe5e5", borde=COLOR["Rojo"])
    boton(fig, 0.73, 0.20, 0.25, "Bienestar Universitario")

    rect(fig, 0, 0.03, 1, 0.12, relleno="#f4f6f9")
    texto(fig, 0.5, 0.075, "Tus datos se usan solo para acompañarte en la asignatura. Puedes retirar tu consentimiento "
                           "en cualquier momento, sin ninguna consecuencia académica.", 12, "#4a5568", xanchor="center",
          yanchor="middle")
    return fig

est_amarillo = c[c.semaforo == "Amarillo"].sort_values("p_riesgo", ascending=False).iloc[0]
pantalla_p4(est_amarillo).show()

## 5. Revisión de sesgos

Las variables demográficas no entran al modelo, pero el modelo podría funcionar peor para algunos grupos, porque la actividad en línea se relaciona con el trabajo, la conectividad o la edad. Por eso comparamos el desempeño por subgrupo con el umbral de alerta amplio (p ≥ 0,40).

In [15]:
test["alerta"] = (test.p_riesgo >= UMBRAL_AMARILLO).astype(int)

def metricas(g):
    neg = (g.riesgo == 0).sum()
    return pd.Series({"n": len(g),
                      "AUC": roc_auc_score(g.riesgo, g.p_riesgo) if g.riesgo.nunique() > 1 else np.nan,
                      "recall": recall_score(g.riesgo, g.alerta),
                      "falsas alarmas": ((g.alerta == 1) & (g.riesgo == 0)).sum() / neg if neg else np.nan})

grupos = {"gender": "Sexo", "age_band": "Edad", "disability": "Discapacidad"}
tablas = []
for var, nombre in grupos.items():
    t = test.groupby(var, observed=True)[["riesgo", "p_riesgo", "alerta"]].apply(metricas)
    t = t.rename(index={"F": "Mujeres", "M": "Hombres", "N": "No", "Y": "Sí", "0-35": "Menos de 35",
                        "35-55": "35 a 55", "55<=": "55 o más"})
    t.index = pd.MultiIndex.from_product([[nombre], t.index.astype(str)])
    tablas.append(t)
sesgos = pd.concat(tablas)
display(sesgos.style.format({"n": "{:,.0f}", "AUC": "{:.2f}", "recall": "{:.2f}", "falsas alarmas": "{:.2f}"}))

fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.08, subplot_titles=list(grupos.values()))
for j, nombre in enumerate(grupos.values(), start=1):
    d = sesgos.loc[nombre]
    fig.add_trace(go.Bar(x=d.index, y=d.recall, name="recall", marker_color=COLOR["azul"], showlegend=j == 1,
                         text=d.recall.map(dec), textposition="outside", cliponaxis=False), row=1, col=j)
    fig.add_trace(go.Bar(x=d.index, y=d["falsas alarmas"], name="falsas alarmas", marker_color="#b0bac6",
                         showlegend=j == 1, text=d["falsas alarmas"].map(dec), textposition="outside",
                         cliponaxis=False), row=1, col=j)
fig.update_yaxes(range=[0, 1.05])
fig.update_layout(height=460, barmode="group", font=FUENTE, margin=dict(t=90, b=80, l=40, r=20),
                  title=dict(text="Recall y falsas alarmas por grupo (alerta p ≥ 0,40)", x=0.01),
                  legend=dict(orientation="h", y=-0.14, x=0.5, xanchor="center"))
fig.show()

**Lectura.**
- La AUC es algo menor para **mujeres** (≈ 0,69) que para hombres (≈ 0,73).
- Los estudiantes de **35 a 55 años** tienen menor recall (≈ 0,77 frente a ≈ 0,84 de los menores de 35): el modelo detecta menos a los adultos que terminan en riesgo. Esto coincide con el sesgo previsto en el diccionario, porque los adultos que trabajan participan distinto en línea.
- Los menores de 35 reciben **más falsas alarmas**.

Todavía **no sabemos si estas diferencias son reales o se deben al azar**.

## 6. Lo que este prototipo no puede decir todavía

1. **No son datos de la UCuenca.** OULAD viene de una universidad a distancia del Reino Unido, con estudiantes adultos y otra organización de cursos. El IPS y los umbrales habría que recalibrarlos con datos de eVirtual.
2. **No demuestra que la herramienta ayude.** El modelo predice el riesgo, pero no sabemos si una tutoría en la semana 4 cambia el resultado. Para eso está la validación pedagógica con grupo de comparación (`06_validacion/Plan_de_validacion.pdf`).
3. **Predecir no es explicar.** Que pocos clics se asocien con reprobar no significa que hacer más clics haga aprobar. El indicador no debe convertirse en una meta para el estudiante.
4. **El desempeño es moderado (AUC ≈ 0,72).** Habrá falsas alarmas y estudiantes que no se detectan. Tampoco sabemos si la diferencia entre la regresión logística y el árbol es significativa, ni cuál es el intervalo de confianza del recall.
5. **Un umbral único no sirve para todos los cursos.** El porcentaje de estudiantes en rojo va de cerca del 10 % a más del 70 % según la asignatura, así que habría que calibrar por curso.
6. **Las diferencias entre grupos (sección 5) no se han probado estadísticamente**, y faltan variables relevantes para la UCuenca, como si el estudiante trabaja o su conectividad.
7. **Faltan los datos que generaría la propia herramienta.** Las intervenciones, el consentimiento y la asistencia no existen en OULAD, por eso P2 y P3 aparecen sin registros.
8. **No evaluamos la usabilidad.** No sabemos si los tutores entienden el semáforo ni si lo usarían cada semana.

---
*Reproducibilidad:* semilla 42; dataset OULAD (UCI, CC BY 4.0); probado con Python 3.14, pandas, scikit-learn y plotly (ver `requirements.txt`).